<a href="https://colab.research.google.com/github/rd2080/Pizza-Data-Analysis/blob/main/Data_Science_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the dataset
file_path = "Pizza_Sale.xlsx"  # Update this if needed
xls = pd.ExcelFile(file_path)

In [ ]:
# Load the pizza_sales sheet
df = pd.read_excel(xls, sheet_name="pizza_sales")

print("\nOriginal Dataset:")
print(df.info())
print(df.head())


Original Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48620 non-null  int64  
 1   order_id           48620 non-null  int64  
 2   pizza_name_id      48604 non-null  object 
 3   quantity           48620 non-null  int64  
 4   order_date         48620 non-null  object 
 5   order_time         48620 non-null  object 
 6   unit_price         48620 non-null  float64
 7   total_price        48613 non-null  float64
 8   pizza_size         48620 non-null  object 
 9   pizza_category     48597 non-null  object 
 10  pizza_ingredients  48607 non-null  object 
 11  pizza_name         48613 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 4.5+ MB
None
   pizza_id  order_id  pizza_name_id  quantity           order_date  \
0         1         1     hawaiian_m         1  2015-01-01 00:

In [ ]:
# Step 1: Handle Missing Values
df.dropna(subset=['pizza_name_id', 'total_price', 'pizza_category', 'pizza_ingredients', 'pizza_name'], inplace=True)

print("\nAfter Handling Missing Values:")
print(df.info())
print(df.head())


After Handling Missing Values:
<class 'pandas.core.frame.DataFrame'>
Index: 48554 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   pizza_id           48554 non-null  int64  
 1   order_id           48554 non-null  int64  
 2   pizza_name_id      48554 non-null  object 
 3   quantity           48554 non-null  int64  
 4   order_date         48554 non-null  object 
 5   order_time         48554 non-null  object 
 6   unit_price         48554 non-null  float64
 7   total_price        48554 non-null  float64
 8   pizza_size         48554 non-null  object 
 9   pizza_category     48554 non-null  object 
 10  pizza_ingredients  48554 non-null  object 
 11  pizza_name         48554 non-null  object 
dtypes: float64(2), int64(3), object(7)
memory usage: 4.8+ MB
None
   pizza_id  order_id  pizza_name_id  quantity           order_date  \
0         1         1     hawaiian_m         1  2015-0

In [ ]:
# Step 2: Convert Data Types
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S', errors='coerce').dt.time

print("\nAfter Converting Data Types:")
print(df.dtypes)
print(df.head())


After Converting Data Types:
pizza_id                      int64
order_id                      int64
pizza_name_id                object
quantity                      int64
order_date           datetime64[ns]
order_time                   object
unit_price                  float64
total_price                 float64
pizza_size                   object
pizza_category               object
pizza_ingredients            object
pizza_name                   object
dtype: object
   pizza_id  order_id  pizza_name_id  quantity order_date order_time  \
0         1         1     hawaiian_m         1 2015-01-01   11:38:36   
1         2         2  classic_dlx_m         1 2015-01-01   11:57:40   
2         3         2  five_cheese_l         1 2015-01-01   11:57:40   
3         4         2    ital_supr_l         1 2015-01-01   11:57:40   
4         5         2     mexicana_m         1 2015-01-01   11:57:40   

   unit_price  total_price pizza_size pizza_category  \
0       13.25        13.25         

In [ ]:
# Step 3: Remove Outliers (Z-score method)
numeric_cols = ['quantity', 'unit_price', 'total_price']
df = df[(np.abs(stats.zscore(df[numeric_cols])) < 3).all(axis=1)]

print("\nAfter Removing Outliers:")
print(df.describe())


After Removing Outliers:
           pizza_id      order_id  quantity                     order_date  \
count  47599.000000  47599.000000   47599.0                          47599   
mean   24337.477720  10713.379357       1.0  2015-06-28 22:43:56.677241088   
min        1.000000      1.000000       1.0            2015-01-01 00:00:00   
25%    12187.500000   5351.500000       1.0            2015-03-30 00:00:00   
50%    24341.000000  10701.000000       1.0            2015-06-28 00:00:00   
75%    36474.500000  16103.500000       1.0            2015-09-29 00:00:00   
max    48620.000000  21350.000000       1.0            2015-12-31 00:00:00   
std    14019.176865   6172.917075       0.0                            NaN   

         unit_price   total_price  
count  47599.000000  47599.000000  
mean      16.478359     16.478359  
min        9.750000      9.750000  
25%       12.750000     12.750000  
50%       16.500000     16.500000  
75%       20.250000     20.250000  
max       25.500000

In [ ]:
# Step 4: Standardize Categorical Values
df['pizza_category'] = df['pizza_category'].str.strip().str.lower()

print("\nAfter Standardizing Categorical Values:")
print(df['pizza_category'].value_counts())


After Standardizing Categorical Values:
pizza_category
classic    14236
supreme    11558
veggie     11231
chicken    10574
Name: count, dtype: int64


In [ ]:
# Step 5: Reshape the Dataset (Aggregate Total Sales per Pizza)
df_reshaped = df.groupby(['pizza_name', 'pizza_size', 'pizza_category'])[['quantity', 'total_price']].sum().reset_index()

print("\nAfter Reshaping the Dataset:")
print(df_reshaped.head())


After Reshaping the Dataset:
                   pizza_name pizza_size pizza_category  quantity  total_price
0  The Barbecue Chicken Pizza          L        chicken       941     19525.75
1  The Barbecue Chicken Pizza          M        chicken       898     15041.50
2  The Barbecue Chicken Pizza          S        chicken       473      6030.75
3          The Big Meat Pizza          S        classic      1715     20580.00
4        The Brie Carre Pizza          S        supreme       469     11091.85


In [ ]:
# Save the cleaned dataset
output_file = "Cleaned_Pizza_Sales.xlsx"
df_reshaped.to_excel(output_file, index=False)

print(f"\nProcessed dataset saved as {output_file}")


Processed dataset saved as Cleaned_Pizza_Sales.xlsx
